In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/09/28 08:27:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [31]:
df_sorted = (
    spark.sql("select * from serving_db.klines")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

In [32]:
df_sorted.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.765|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.883|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.

In [33]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("ema12", types.DoubleType(), True),
    types.StructField("ema26", types.DoubleType(), True),
    types.StructField("macd", types.DoubleType(), True),
    types.StructField("signal", types.DoubleType(), True),
    types.StructField("histogram", types.DoubleType(), True)
])

In [34]:
def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10 ** decimals
    return float(int(x * factor + 0.5)) / factor

def calc_ema(value, state):
    if value is None:
        return None
    prev, buffer, period, k = state["prev"], state["buffer"], state["period"], state["k"]
    if prev is None:
        buffer.append(value)
        if len(buffer) == period:
            ema = sum(buffer) / len(buffer)
        else:
            ema = None
    else:
        ema = (value - prev) * k + prev

    state["prev"] = ema
    return ema

def rounded(dec):
    return float(dec.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

def ema_in_chunks(iterator):
    ema_configs = {
        "ema12": {"period": 12, "k": 2 / (12 + 1), "prev": None, "buffer": []},
        "ema26": {"period": 26, "k": 2 / (26 + 1), "prev": None, "buffer": []},
        "signal": {"period": 9, "k": 2 / (9 + 1), "prev": None, "buffer": []}
    }

    for pdf in iterator:
        ema12, ema26, macd_line, signal_line, histogram_line = [], [], [], [], []
        for p in pdf["close_price"]:
            price = float(p)
            # step 1: compute ema12, ema26
            e12 = calc_ema(price, ema_configs["ema12"])
            ema12.append(round_half_up(e12, 2) if e12 is not None else None)
            e26 = calc_ema(price, ema_configs["ema26"])
            ema26.append(round_half_up(e26, 2) if e26 is not None else None)
            
            # step 2: MACD line = ema12 - ema26
            macd = e12 - e26 if e12 is not None and e26 is not None else None
            macd_line.append(round_half_up(macd, 2) if macd is not None else None)
    
            # step 3: Signal line (ema9 of MACD)
            signal = calc_ema(macd, ema_configs["signal"])
            signal_line.append(round_half_up(signal, 2) if signal is not None else None)
    
            # step 4: Histogram = MACD - Signal
            histogram = macd - signal if macd is not None and signal is not None else None
            histogram_line.append(round_half_up(histogram, 2) if histogram is not None else None)
            
        pdf["ema12"] = ema12
        pdf["ema26"] = ema26
        pdf["macd"] = macd_line
        pdf["signal"] = signal_line
        pdf["histogram"] = histogram_line

        # final order
        pdf = pdf[[*pdf.columns[:-5], "ema12", "ema26", "macd", "signal", "histogram"]]
        yield pdf

In [35]:
df = df_sorted.mapInPandas(ema_in_chunks, schema)

In [36]:
df.writeTo("serving_db.macd").tableProperty("format-version", "2").createOrReplace()

In [38]:
spark.sql("""
select * from serving_db.macd
""").orderBy("group_id", ascending=False).show()

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+---------+-------+-------+---------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|    ema12|    ema26|   macd| signal|histogram|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+---------+-------+-------+---------+
| 1948991|2025-08-01 23:45:00|1754091900178327| 113344.67|  113400.0|113211.26|  113297.93|138.817|1754092799895345|113479.71|113777.93|-298.21|-317.36|    19.15|
| 1948990|2025-08-01 23:30:00|1754091000512531| 113549.44| 113549.44|113224.37|  113344.67|115.198|1754091899935216|113512.76|113816.33|-303.56|-322.15|    18.59|
| 1948989|2025-08-01 23:15:00|1754090100250089| 113311.32| 113560.46|113238.69|  113549.44|100.521|1754090999730086|113543.32|113854.07|-310.73| -326.8|    16.07|
| 1948988|2025-08-01 2

In [40]:
spark.sql("""
select count(*) as rows from serving_db.macd
""").show()

+----+
|rows|
+----+
|  96|
+----+

